# 🦙 Llama-3.1-8B SAE — Kaggle Free Tier

**TopK SAE on Llama-3.1-8B residual stream, L16. ~5–6 h on 2×T4 (or P100 16 GB with offload).**

---

| Tier | Hardware | Model | Tokens | Wall time | Cost |
|------|----------|-------|--------|-----------|------|
| 1 — Hobbyist | Colab T4 | Gemma-2-2B | 20M | ~30 min | $0 |
| 2 — Explorer (Qwen3.5-4B) | Kaggle 2×T4 | Qwen3.5-4B | 150M | ~4–5 h | $0 |
| **2b — Explorer (this)** | **Kaggle 2×T4** | **Llama-3.1-8B** | **100M** | **~5–6 h** | **$0** |
| 3 — Paper-grade | Vast.ai B200 | Qwen3.5-9B / Gemma-4 | 1B+ | ~22 h | ~$30 |

Llama-3.1-8B is the most-downloaded dense 8B on the Hub. SAE features trained on it transfer well as a starting point for interpretability work on any Llama-family derivative (Nemotron, Hermes, Tulu, Dolphin, etc.).

**License**: Llama-3.1 is governed by Meta's Llama 3.1 Community License. You must accept it on the model page first:
<https://huggingface.co/meta-llama/Llama-3.1-8B>. Your HF token must be linked to an account that has been granted access, otherwise the download at Cell 8 will 403.

**Kaggle quota**: Free tier gives 30 h/week of accelerator time. This notebook consumes ~5–6 h of that budget.

**Runtime**: Kaggle kernels hard-cap at 9 h of continuous wall time and can be pre-empted earlier. Every 10M tokens we upload a resume file to HF. If the kernel dies, just restart and rerun all cells — it picks up where it left off.

In [ ]:
# Cell 2 — Install. Pinned versions; NO flash-attn (T4 is SM 7.5, flash-attn is SM 8.0+).
import sys, subprocess

def pip(*a):
    return subprocess.run([sys.executable, '-m', 'pip', *a], check=False)

pip('install', '-q',
    'transformers==4.57.1',
    'accelerate==1.12.0',
    'datasets==4.0.0',
    'safetensors==0.4.5',
    'huggingface_hub==1.5.0',
    'einops==0.8.1',
    'sentencepiece',
    'tokenizers',
    'protobuf')

import transformers, accelerate, datasets, huggingface_hub
print(f'transformers {transformers.__version__}')
print(f'accelerate   {accelerate.__version__}')
print(f'datasets     {datasets.__version__}')
print(f'hf_hub       {huggingface_hub.__version__}')

## Config — 100M tokens in ~5–6 h on 2×T4

Llama-3.1-8B is meaningfully heavier than Qwen3.5-4B — 8.03B vs 4.02B params, `D_MODEL=4096` vs 2560 — so we shrink the budget from 150M → 100M tokens and drop `FWD_BATCH` from 2 → 1.

- `N_FEATURES = 65536` = 16× expansion over `D_MODEL = 4096`
- `K_TOPK = 128` — ~0.2 % active features
- `K_AUX = 2048` (= d/2) with `ALPHA_AUX = 1/32` — dead-feature rescue
- `DEAD_TOKENS = 10M` — a feature is "dead" if unused for this many tokens
- `FWD_BATCH = 1 × SEQ_LEN 1024` on the model's device; SAE lives on the other GPU
- `LAYER = 16` — middle of Llama-3.1-8B's 32-layer stack

**100M tokens / (FWD_BATCH 1 × SEQ_LEN 1024) = ~98k forward passes ≈ 320 min at ~5 passes/sec on T4 = ~5.3 h.**

If you're on a single P100 (16 GB) instead of 2×T4, the same config works — `device_map='auto'` will still place the SAE + model in the same visible device, VE metric is unchanged, wall time is ~5 h (P100 has slightly better fp32 throughput than T4 but no bf16 acceleration).

In [ ]:
# Cell 4 — Config
MODEL_ID       = 'meta-llama/Llama-3.1-8B'
LAYER          = 16                 # middle of 32-layer stack
D_MODEL        = 4096
N_FEATURES     = 65536              # 16x expansion
K_TOPK         = 128
K_AUX          = 2048               # d/2
ALPHA_AUX      = 1/32
DEAD_TOKENS    = 10_000_000
TOKEN_BUDGET   = 100_000_000        # conservative for Kaggle 8B
SEQ_LEN        = 1024
FWD_BATCH      = 1                  # 8B is tight on T4; 1 sequence per forward
BATCH_SIZE     = 4096               # SAE optimizer batch (token-level)
LR_PEAK        = 2e-4
LR_FLOOR       = 6e-5
WARMUP_STEPS   = 3000
CKPT_EVERY_TOK = 10_000_000         # HF checkpoint cadence (kernel-kill safe)

HF_USERNAME    = 'YOUR_HF_USERNAME'  # <-- EDIT
HF_REPO        = f'{HF_USERNAME}/llama3-8b-sae-L16'

print(f'Target: {TOKEN_BUDGET/1e6:.0f}M tokens into L{LAYER} SAE of {N_FEATURES} features, k={K_TOPK}')
print(f'Checkpointing to hf://{HF_REPO} every {CKPT_EVERY_TOK/1e6:.0f}M tokens')

## Auth — Kaggle Secrets

1. Kaggle → Add-ons → Secrets → **Add Secret** → name: `HF_TOKEN`, value: a write-scoped HF token.
2. Attach it to this notebook (the toggle next to the secret name).

The token's account must have been granted access to `meta-llama/Llama-3.1-8B` — apply at the model page, Meta usually approves within an hour.

In [ ]:
# Cell 6 — HF auth via Kaggle Secrets
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, HfApi, create_repo

hf_token = UserSecretsClient().get_secret('HF_TOKEN')
login(token=hf_token)
api = HfApi(token=hf_token)

try:
    create_repo(HF_REPO, repo_type='model', exist_ok=True, token=hf_token)
    print(f'Repo ready: https://huggingface.co/{HF_REPO}')
except Exception as e:
    print(f'repo warn: {e}')

## Load Llama-3.1-8B

Llama is a standard dense transformer — plain `AutoModelForCausalLM`, no trust_remote_code, no multimodal wrapper. We load in `bfloat16` (T4 emulates it, P100 emulates it — both work, just no tensor-core speedup) with `attn_implementation='sdpa'`. **NEVER flash-attn on Kaggle** — T4 is Turing / SM 7.5 and P100 is Pascal / SM 6.0, both below flash-attn's SM 8.0 floor.

`device_map='auto'` will spread the 16 GB of bf16 weights across both T4s (roughly 8 GB each), leaving ~8 GB free on each for activations + SAE. On a single P100 (16 GB) everything lives there; we rely on activation streaming + no_grad to keep VRAM manageable.

**First run downloads ~16 GB of safetensors — budget ~3–5 min on Kaggle's network.**

In [ ]:
# Cell 8 — Model load
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print(f'torch {torch.__version__} — {torch.cuda.device_count()} GPU(s)')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  [{i}] {p.name}  {p.total_memory/1e9:.1f} GB  SM{p.major}.{p.minor}')

tok = AutoTokenizer.from_pretrained(MODEL_ID, token=hf_token)
if tok.pad_token_id is None:
    tok.pad_token_id = tok.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,             # NEVER torch_dtype= (deprecated in transformers 5.x)
    device_map='auto',
    attn_implementation='sdpa',       # NEVER flash-attn on T4/P100
    token=hf_token,
)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

torch.cuda.empty_cache()
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f'  GPU{i} after model load: {(total-free)/1e9:.1f} / {total/1e9:.1f} GB used')

print(f'model class: {model.__class__.__name__}')
print(f'config model_type: {model.config.model_type}  '
      f'hidden_size={model.config.hidden_size}  n_layers={model.config.num_hidden_layers}')
assert model.config.hidden_size == D_MODEL, \
    f'D_MODEL mismatch: config says {model.config.hidden_size}, config cell says {D_MODEL}'

## Residual capture via forward hook on `model.model.layers[16]`

Llama uses the standard `model.model.layers[N]` path (no multimodal nesting, no `.language_model.`). A decoder block's output is a tuple `(hidden_states, ...)` — we grab element 0, which is the residual stream after that layer's attention + MLP have written to it.

We avoid `output_hidden_states=True` here because it allocates a list of all 33 tensors (embeddings + 32 layers) every forward — wasteful when we only need one.

### Corpus — FineWeb-Edu `sample-10BT`
Broad, high-quality web, Common-Crawl filtered for educational content. 10B-token shuffle is more than enough for 100M tokens of training.


In [ ]:
# Cell 10 — Register forward hook on layer 16
_captured = {}

def _layer_hook(module, inputs, output):
    # Llama decoder blocks return either a tuple (hidden_states, ...) or a bare tensor
    # depending on the transformers version. Handle both.
    h = output[0] if isinstance(output, tuple) else output
    _captured['h'] = h

target_layer = model.model.layers[LAYER]
hook_handle = target_layer.register_forward_hook(_layer_hook)
print(f'Hook attached to model.model.layers[{LAYER}] '
      f'({target_layer.__class__.__name__}) on device {next(target_layer.parameters()).device}')

## Corpus streaming + activation generator

FineWeb-Edu streamed with a 2000-doc shuffle buffer, packed end-to-end into `SEQ_LEN=1024` sequences with EOS separators. Each activation batch is `(FWD_BATCH * SEQ_LEN, D_MODEL) = (1024, 4096)` fp32 tensors, moved onto the SAE's GPU.

In [ ]:
# Cell 12 — Streaming corpus + activation generator
import itertools, random
from datasets import load_dataset

def open_stream(seed=0):
    try:
        ds = load_dataset(
            'HuggingFaceFW/fineweb-edu',
            name='sample-10BT',
            split='train',
            streaming=True,
        ).shuffle(seed=seed, buffer_size=2000)
    except Exception as e:
        print(f'fineweb-edu sample-10BT failed ({e}); falling back to CC-MAIN-2024-10')
        ds = load_dataset(
            'HuggingFaceFW/fineweb-edu',
            name='CC-MAIN-2024-10',
            split='train',
            streaming=True,
        ).shuffle(seed=seed, buffer_size=2000)
    return ds

def text_stream(seed=0):
    rng = random.Random(seed)
    ds = open_stream(seed=seed)
    it = iter(ds)
    while True:
        try:
            sample = next(it)
        except StopIteration:
            ds = open_stream(seed=seed + rng.randint(1, 1_000_000))
            it = iter(ds)
            continue
        txt = sample.get('text', '')
        if isinstance(txt, str) and len(txt) > 64:
            yield txt

def tokenize_pack(text_iter, seq_len=SEQ_LEN):
    """Pack short docs end-to-end into fixed-length sequences."""
    buf = []
    for txt in text_iter:
        ids = tok(txt, add_special_tokens=False).input_ids
        buf.extend(ids)
        buf.append(tok.eos_token_id or 0)
        while len(buf) >= seq_len:
            yield buf[:seq_len]
            buf = buf[seq_len:]

def batched(iterable, n):
    it = iter(iterable)
    while True:
        chunk = list(itertools.islice(it, n))
        if not chunk:
            return
        yield chunk

# Decide SAE device: if 2 GPUs, pin SAE to the one NOT holding layer 16.
# With device_map='auto' Llama-8B spreads across both T4s; we pick the less-loaded one.
_layer_dev = next(target_layer.parameters()).device
if torch.cuda.device_count() > 1:
    dev_sae = torch.device('cuda:1') if _layer_dev.index == 0 else torch.device('cuda:0')
else:
    dev_sae = _layer_dev
print(f'layer {LAYER} lives on {_layer_dev}; SAE will live on {dev_sae}')

@torch.no_grad()
def activation_stream(seed=0):
    """Yields (FWD_BATCH*SEQ_LEN, D_MODEL) fp32 residual tensors on dev_sae."""
    dev_in = next(model.get_input_embeddings().parameters()).device
    seqs = tokenize_pack(text_stream(seed=seed))
    for chunk in batched(seqs, FWD_BATCH):
        ids = torch.tensor(chunk, dtype=torch.long, device=dev_in)
        _ = model(input_ids=ids, use_cache=False)   # hook populates _captured['h']
        h = _captured['h']
        h = h.reshape(-1, h.shape[-1]).to(dev_sae, dtype=torch.float32)
        _captured.clear()
        yield h

# Smoke test — one batch
it = activation_stream(seed=123)
sample_batch = next(it)
print(f'sample activation batch: {tuple(sample_batch.shape)}  dtype={sample_batch.dtype}  dev={sample_batch.device}')
print(f'mean={sample_batch.mean().item():+.4f}  std={sample_batch.std().item():.4f}  '
      f'abs_max={sample_batch.abs().max().item():.2f}')
del it, sample_batch
torch.cuda.empty_cache()

## TopK SAE + AuxK (Gao et al., 2024)

Standard OpenAI-style TopK SAE:

- Encoder: `Linear(d → n) → bias → TopK(k)` — everything else is zeroed.
- Decoder: `Linear(n → d)` with unit-norm columns (re-normalized after each step).
- AuxK: among features that haven't fired in `DEAD_TOKENS`, take the top `K_AUX` pre-activations and use them to explain the residual error. Gradient flows back — this is how dead features come back to life.
- Geometric-median init for `b_dec` (Weiszfeld) to put the decoder bias near the data mean.

SAE weights stay in **fp32** even though the model is bf16 — Adam moments want fp32 for stability and T4/P100 don't accelerate bf16 matmuls anyway.

In [ ]:
# Cell 14 — TopK SAE with AuxK
import torch
import torch.nn as nn
import torch.nn.functional as F

class TopKSAE(nn.Module):
    def __init__(self, d=D_MODEL, n=N_FEATURES, k=K_TOPK, k_aux=K_AUX, dead_tokens=DEAD_TOKENS):
        super().__init__()
        self.d, self.n, self.k, self.k_aux = d, n, k, k_aux
        self.dead_tokens = dead_tokens

        self.W_enc = nn.Parameter(torch.empty(d, n))
        self.b_enc = nn.Parameter(torch.zeros(n))
        self.W_dec = nn.Parameter(torch.empty(n, d))
        self.b_dec = nn.Parameter(torch.zeros(d))

        nn.init.kaiming_uniform_(self.W_enc, a=5**0.5)
        with torch.no_grad():
            self.W_dec.copy_(self.W_enc.t().contiguous())
            self.renorm_decoder()

        self.register_buffer('last_fired', torch.zeros(n, dtype=torch.long))
        self.register_buffer('tokens_seen', torch.zeros(1, dtype=torch.long))

    @torch.no_grad()
    def renorm_decoder(self):
        nrm = self.W_dec.norm(dim=1, keepdim=True).clamp_min(1e-8)
        self.W_dec.div_(nrm)

    @torch.no_grad()
    def set_b_dec_geomedian(self, samples, iters=50, eps=1e-5):
        """Weiszfeld iteration. samples: (N, d) fp32."""
        x = samples.to(self.b_dec.device)
        mu = x.mean(0)
        for _ in range(iters):
            d = (x - mu).norm(dim=1).clamp_min(eps)
            w = 1.0 / d
            mu_new = (w[:, None] * x).sum(0) / w.sum()
            if (mu_new - mu).norm() < eps:
                break
            mu = mu_new
        self.b_dec.copy_(mu)

    def encode_pre(self, x):
        return (x - self.b_dec) @ self.W_enc + self.b_enc

    def forward(self, x):
        """x: (B, d)  → (recon, aux_recon, z, topk_idx, pre)"""
        pre = self.encode_pre(x)                             # (B, n)
        topk_val, topk_idx = pre.topk(self.k, dim=-1)
        z = torch.zeros_like(pre)
        z.scatter_(-1, topk_idx, F.relu(topk_val))
        recon = z @ self.W_dec + self.b_dec

        aux_recon = None
        if self.training and self.k_aux > 0:
            dead_mask = (self.last_fired >= self.dead_tokens)       # (n,)
            n_dead = int(dead_mask.sum().item())
            if n_dead > 0:
                k_aux_eff = min(self.k_aux, n_dead)
                pre_dead = pre.masked_fill(~dead_mask, float('-inf'))
                aux_val, aux_idx = pre_dead.topk(k_aux_eff, dim=-1)
                z_aux = torch.zeros_like(pre)
                z_aux.scatter_(-1, aux_idx, F.relu(aux_val))
                aux_recon = z_aux @ self.W_dec

        return recon, aux_recon, z, topk_idx, pre

    @torch.no_grad()
    def update_fire_counter(self, topk_idx, batch_tokens):
        self.last_fired += batch_tokens
        fired = torch.unique(topk_idx)
        self.last_fired[fired] = 0
        self.tokens_seen += batch_tokens

    @torch.no_grad()
    def dead_count(self):
        return int((self.last_fired >= self.dead_tokens).sum().item())


print('TopKSAE defined.')

## Initialize + resume from HF checkpoint + training loop

Kaggle kernels die. To survive that we:
1. Upload `sae_L16_resume.pt` (weights + optimizer + step + tokens) every `CKPT_EVERY_TOK = 10M` tokens.
2. On (re)start, try to download it and pick up where we left off.
3. On fresh start, init `b_dec` with a geometric median over ~64 k activations.

**If the kernel gets killed** (9 h cap, network drop, pre-emption): Kaggle → Run → Restart & Run All. The resume file in the HF repo will be picked up automatically.

Loss:

$$ \mathcal{L} = \lVert x - \hat{x} \rVert_2^2 + \alpha_\text{aux} \lVert (x - \hat{x})_\text{stop-grad} - \hat{x}_\text{aux} \rVert_2^2 $$

In [ ]:
# Cell 16 — Full training loop: init/resume → train → checkpoint → final upload → validate
import math, os, json, io, time
from pathlib import Path
from safetensors.torch import save_file as save_safetensors
from huggingface_hub import hf_hub_download, upload_file, delete_file
from tqdm.auto import tqdm

TMP = Path('/kaggle/working/ckpt'); TMP.mkdir(exist_ok=True)
RESUME_NAME  = f'sae_L{LAYER}_resume.pt'
LATEST_NAME  = f'sae_L{LAYER}_latest.safetensors'

# --- Instantiate SAE + optimizer ---
sae = TopKSAE().to(dev_sae, dtype=torch.float32)
optim = torch.optim.Adam(sae.parameters(), lr=LR_PEAK, betas=(0.9, 0.999), eps=1e-8)

TOTAL_STEPS = max(1, TOKEN_BUDGET // BATCH_SIZE)
def lr_at(step):
    if step < WARMUP_STEPS:
        return LR_PEAK * step / max(1, WARMUP_STEPS)
    prog = (step - WARMUP_STEPS) / max(1, TOTAL_STEPS - WARMUP_STEPS)
    prog = min(1.0, max(0.0, prog))
    return LR_FLOOR + 0.5 * (LR_PEAK - LR_FLOOR) * (1 + math.cos(math.pi * prog))

state = {'step': 0, 'tokens_seen': 0}
resumed = False
try:
    local = hf_hub_download(repo_id=HF_REPO, filename=RESUME_NAME, token=hf_token)
    ckpt = torch.load(local, map_location='cpu', weights_only=False)
    sae.load_state_dict(ckpt['sae'])
    optim.load_state_dict(ckpt['optim'])
    state['step'] = int(ckpt.get('step', 0))
    state['tokens_seen'] = int(ckpt.get('tokens_seen', 0))
    sae.to(dev_sae, dtype=torch.float32)
    print(f'Resumed from step {state["step"]}, tokens_seen {state["tokens_seen"]/1e6:.1f}M')
    resumed = True
except Exception as e:
    print(f'No resume file on HF ({type(e).__name__}); fresh init with geometric median.')

if not resumed:
    geo_batches = []
    n_needed = 65536
    n_have = 0
    it = activation_stream(seed=42)
    while n_have < n_needed:
        b = next(it)
        geo_batches.append(b)
        n_have += b.shape[0]
    gm_samples = torch.cat(geo_batches, dim=0)[:n_needed]
    sae.set_b_dec_geomedian(gm_samples, iters=50)
    print(f'b_dec initialised via geom-median on {n_needed} tokens — '
          f'norm={sae.b_dec.norm().item():.3f}')
    del geo_batches, gm_samples, it
    torch.cuda.empty_cache()

# --- Checkpoint helpers ---
def save_weights_safetensors(sae, path):
    sd = {k: v.detach().cpu().contiguous() for k, v in sae.state_dict().items()
          if not k.startswith(('last_fired', 'tokens_seen'))}
    save_safetensors(sd, str(path))

def save_resume(path, sae, optim, step, tokens_seen):
    torch.save({
        'sae': sae.state_dict(),
        'optim': optim.state_dict(),
        'step': step,
        'tokens_seen': tokens_seen,
    }, path)

def push_checkpoint(sae, optim, step, tokens_seen, is_final=False):
    w_path = TMP / LATEST_NAME
    r_path = TMP / RESUME_NAME
    cfg_path = TMP / 'cfg.json'
    save_weights_safetensors(sae, w_path)
    cfg = dict(
        model_id=MODEL_ID, layer=LAYER, d_model=D_MODEL, n_features=N_FEATURES,
        k_topk=K_TOPK, k_aux=K_AUX, alpha_aux=ALPHA_AUX,
        dead_tokens=DEAD_TOKENS, seq_len=SEQ_LEN,
        batch_size=BATCH_SIZE, token_budget=TOKEN_BUDGET,
        step=step, tokens_seen=tokens_seen, final=is_final,
    )
    cfg_path.write_text(json.dumps(cfg, indent=2))
    upload_file(path_or_fileobj=str(w_path), path_in_repo=LATEST_NAME,
                repo_id=HF_REPO, token=hf_token)
    upload_file(path_or_fileobj=str(cfg_path), path_in_repo='cfg.json',
                repo_id=HF_REPO, token=hf_token)
    if not is_final:
        save_resume(r_path, sae, optim, step, tokens_seen)
        upload_file(path_or_fileobj=str(r_path), path_in_repo=RESUME_NAME,
                    repo_id=HF_REPO, token=hf_token)

# --- Training loop ---
step = state['step']
tokens_seen = state['tokens_seen']
next_ckpt_at = ((tokens_seen // CKPT_EVERY_TOK) + 1) * CKPT_EVERY_TOK

pbar = tqdm(total=TOKEN_BUDGET, initial=tokens_seen, desc='training', unit='tok',
            unit_scale=True, smoothing=0.05)

acts_iter = activation_stream(seed=1000 + step)
acts_buf = torch.empty(0, D_MODEL, device=dev_sae, dtype=torch.float32)
running = {'recon': 0.0, 'aux': 0.0, 've': 0.0, 'n': 0}
t0 = time.time()

try:
    sae.train()
    while tokens_seen < TOKEN_BUDGET:
        while acts_buf.shape[0] < BATCH_SIZE:
            chunk = next(acts_iter)
            acts_buf = torch.cat([acts_buf, chunk], dim=0)
        x = acts_buf[:BATCH_SIZE]
        acts_buf = acts_buf[BATCH_SIZE:]

        lr = lr_at(step)
        for g in optim.param_groups:
            g['lr'] = lr

        recon, aux_recon, z, topk_idx, pre = sae(x)
        err = x - recon
        recon_loss = err.pow(2).mean()
        if aux_recon is not None:
            aux_err = err.detach() - aux_recon
            aux_loss = aux_err.pow(2).mean()
            loss = recon_loss + ALPHA_AUX * aux_loss
        else:
            aux_loss = torch.tensor(0.0, device=dev_sae)
            loss = recon_loss

        optim.zero_grad(set_to_none=True)
        loss.backward()
        # Strip out the component of W_dec.grad parallel to each decoder column
        # (keeps columns unit-norm in the projected-gradient sense).
        with torch.no_grad():
            if sae.W_dec.grad is not None:
                proj = (sae.W_dec.grad * sae.W_dec).sum(dim=1, keepdim=True)
                sae.W_dec.grad.sub_(proj * sae.W_dec)
        optim.step()
        with torch.no_grad():
            sae.renorm_decoder()
            sae.update_fire_counter(topk_idx, batch_tokens=BATCH_SIZE)

        with torch.no_grad():
            var_x = x.var(unbiased=False).clamp_min(1e-8)
            ve = 1.0 - err.pow(2).mean() / var_x
            running['recon'] += recon_loss.item()
            running['aux']   += aux_loss.item() if aux_recon is not None else 0.0
            running['ve']    += ve.item()
            running['n']     += 1

        step += 1
        tokens_seen += BATCH_SIZE
        pbar.update(BATCH_SIZE)

        if step % 25 == 0:
            n = running['n']
            pbar.set_postfix(
                lr=f'{lr:.2e}',
                recon=f'{running["recon"]/n:.4f}',
                aux=f'{running["aux"]/n:.4f}',
                ve=f'{running["ve"]/n:.3f}',
                dead=sae.dead_count(),
                L0=K_TOPK,
            )
            running = {'recon': 0.0, 'aux': 0.0, 've': 0.0, 'n': 0}

        if tokens_seen >= next_ckpt_at:
            print(f'\n[ckpt @ {tokens_seen/1e6:.1f}M tokens, step {step}] uploading…')
            try:
                push_checkpoint(sae, optim, step, tokens_seen, is_final=False)
                print('[ckpt] done.')
            except Exception as e:
                print(f'[ckpt] upload failed, will retry next window: {e}')
            next_ckpt_at += CKPT_EVERY_TOK

finally:
    pbar.close()
    elapsed = time.time() - t0
    print(f'elapsed {elapsed/3600:.2f} h  —  tokens_seen {tokens_seen/1e6:.1f}M  —  step {step}')

# --- Final upload ---
print('Uploading final weights…')
try:
    push_checkpoint(sae, optim, step, tokens_seen, is_final=True)
    print('Final weights uploaded.')
except Exception as e:
    print(f'final upload error: {e}')

try:
    delete_file(path_in_repo=RESUME_NAME, repo_id=HF_REPO, token=hf_token)
    print(f'{RESUME_NAME} deleted from repo.')
except Exception as e:
    print(f'resume delete warn: {e}')

# --- Held-out validation on 500k fresh tokens ---
print('\nValidation over 500k held-out tokens…')
sae.eval()
val_tokens = 500_000
val_seen = 0
val_recon_sse = 0.0
val_var_sum  = 0.0
L0_sum = 0.0
fired_ever = torch.zeros(N_FEATURES, dtype=torch.bool, device=dev_sae)

val_iter = activation_stream(seed=99999)
with torch.no_grad():
    while val_seen < val_tokens:
        chunk = next(val_iter)
        x = chunk[:min(chunk.shape[0], val_tokens - val_seen)]
        recon, _, z, topk_idx, _ = sae(x)
        err = x - recon
        val_recon_sse += err.pow(2).sum().item()
        val_var_sum   += (x - x.mean(0, keepdim=True)).pow(2).sum().item()
        L0_sum        += (z > 0).float().sum(dim=-1).sum().item()
        fired_ever[torch.unique(topk_idx)] = True
        val_seen += x.shape[0]

ve_val   = 1.0 - val_recon_sse / max(1e-9, val_var_sum)
L0_val   = L0_sum / max(1, val_seen)
dead_val = int((~fired_ever).sum().item())

report = dict(
    model_id=MODEL_ID, layer=LAYER,
    tokens_trained=tokens_seen, steps=step,
    val_tokens=val_seen,
    val_variance_explained=ve_val,
    val_L0=L0_val,
    val_dead_features=dead_val,
    val_dead_frac=dead_val / N_FEATURES,
    k_topk=K_TOPK, n_features=N_FEATURES,
)
print(json.dumps(report, indent=2))

rpath = TMP / 'val_report.json'
rpath.write_text(json.dumps(report, indent=2))
try:
    upload_file(path_or_fileobj=str(rpath), path_in_repo='val_report.json',
                repo_id=HF_REPO, token=hf_token)
    print(f'val_report.json uploaded to https://huggingface.co/{HF_REPO}')
except Exception as e:
    print(f'val report upload warn: {e}')

# Cleanup — free model VRAM so the kernel can shut down cleanly
try:
    hook_handle.remove()
except Exception:
    pass
del model
torch.cuda.empty_cache()
print('\nDone. Next tier: Vast.ai B200 for 1B tokens on Qwen3.5-9B or Gemma-4.')